# DDXPlus Disease Prediction — ML Model Analysis
## Medical Symptom-to-Disease Classifier

### What is DDXPlus?
DDXPlus is a large-scale synthetic medical dataset introduced at NeurIPS 2022 (Fansi Tchango et al., *DDXPlus: A New Dataset for Automatic Medical Diagnosis*). It was generated from a clinically validated medical knowledge base and contains structured patient cases, each with a set of reported symptoms (evidences), demographic information, and a confirmed ground-truth pathology.

**Dataset statistics:**
- ~1.3 million synthetic patient cases across train / validate / test splits
- 223 symptom/evidence codes covering chief complaints, associated symptoms, and antecedents
- 49 disease classes ranging from common infections to more complex conditions
- 3 demographic features per patient: age (integer), sex (M/F), and a differential-diagnosis column with ordered disease probabilities

### What this notebook covers
1. **Section 2** — Load DDXPlus from HuggingFace; explore the data schema  
2. **Section 3** — Exploratory data analysis (class balance, symptom frequency, demographics)  
3. **Section 4** — Feature engineering: multi-hot symptom encoding + demographic features  
4. **Section 5** — Train an XGBoost multi-class classifier with early stopping  
5. **Section 6** — Evaluate: Top-k accuracy, Macro F1, Brier score, confusion matrix  
6. **Section 7** — Explainability with SHAP: global feature importance + per-disease analysis  
7. **Section 8** — Save model artifacts for use in the production assistant  
8. **Section 9** — Inference demo mirroring the runtime flow  
9. **Section 10** — Summary and limitations  

### Relationship to the main project
This notebook is the **data science documentation** of the ML component used inside the Hybrid LLM+RAG+ML Medical Assistant. The artifacts produced in Section 8 are loaded directly by `ml_model/predict.py` in the production codebase. The inference logic in Section 9 mirrors what `symptom_parser.py` + `ml_model/predict.py` do at runtime.

> **License note:** DDXPlus is released for research use only. This notebook is a portfolio/educational artifact and has not been validated for clinical use.

> **Citation:** Fansi Tchango, A., Goel, R., Wen, Z., Martel, J., & Ghosn, J. (2022). DDXPlus: A New Dataset for Automatic Medical Diagnosis. *NeurIPS 2022 Datasets and Benchmarks Track*.

## 1. Environment Setup
Install all required libraries and set the global random seed for reproducibility.

In [ ]:
!pip install -q xgboost scikit-learn shap datasets pandas matplotlib seaborn imbalanced-learn

In [ ]:
import ast
import json
import os
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from datasets import load_dataset
from sklearn.metrics import (
    brier_score_loss,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Environment ready.')

## 2. Loading DDXPlus

### Data schema
Each row in the CSV represents one synthetic patient case. The columns we use:

| Column | Type | Description |
|--------|------|-------------|
| `EVIDENCES` | string | Python-list literal of evidence codes present for this patient, e.g. `['E_1', 'E_14_@_V_0']`. Codes with `_@_V_` suffix encode a discrete value (e.g. severity level). |
| `PATHOLOGY` | string | Ground-truth disease label (one of 49 classes). |
| `DIFFERENTIAL_DIAGNOSIS` | string | Ordered list of `(disease, probability)` pairs from the knowledge base — useful for ranking model targets. |
| `AGE` | int | Patient age in years. |
| `SEX` | str | `'M'` or `'F'`. |

We also load two vocabulary JSONs:
- **`release_evidences.json`** — maps every evidence code to its English question text, data type, and possible values.
- **`release_conditions.json`** — maps every disease to its associated symptom set and severity information.

In [ ]:
print('Loading DDXPlus from HuggingFace (cached after first run) ...')
ds = load_dataset('aai530-group6/ddxplus')
train_df = pd.DataFrame(ds['train'])
val_df   = pd.DataFrame(ds['validate'])
test_df  = pd.DataFrame(ds['test'])

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
train_df.head(3)

In [ ]:
# The JSON files are bundled with the HuggingFace dataset repo.
# If running locally, point these paths at data/raw/ddxplus/.
from datasets import load_dataset as _lds

# Attempt to locate the cached files; fall back to fetching from the repo.
try:
    import huggingface_hub
    _repo = 'aai530-group6/ddxplus'
    _ev_path = huggingface_hub.hf_hub_download(_repo, 'release_evidences.json', repo_type='dataset')
    _cond_path = huggingface_hub.hf_hub_download(_repo, 'release_conditions.json', repo_type='dataset')
except Exception:
    _ev_path = 'release_evidences.json'
    _cond_path = 'release_conditions.json'

with open(_ev_path, encoding='utf-8') as f:
    evidences = json.load(f)
with open(_cond_path, encoding='utf-8') as f:
    conditions = json.load(f)

print(f'Evidence codes: {len(evidences)}')
print(f'Disease conditions: {len(conditions)}')

# Preview one evidence entry
sample_code = list(evidences.keys())[0]
print(f'\nSample evidence entry ({sample_code}):')
print(json.dumps(evidences[sample_code], indent=2))

## 3. Exploratory Data Analysis

Before building anything, we need to understand the data. Three key questions:
1. **Class balance** — are all 49 diseases equally represented, or is there a long tail?
2. **Symptom frequency** — which symptoms appear most often, and how many symptoms does an average patient have?
3. **Demographics** — is the age/sex distribution roughly uniform, or skewed?

These observations directly drive modeling decisions (class weighting, feature importance expectations).

In [ ]:
disease_counts = train_df['PATHOLOGY'].value_counts()

plt.figure(figsize=(16, 6))
disease_counts.plot(kind='bar', color='steelblue')
plt.title('Disease Class Distribution in Training Set', fontsize=14)
plt.xlabel('Disease')
plt.ylabel('Patient Count')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

print(f'Most common : {disease_counts.index[0]} ({disease_counts.iloc[0]:,})')
print(f'Least common: {disease_counts.index[-1]} ({disease_counts.iloc[-1]:,})')
print(f'Imbalance ratio: {disease_counts.iloc[0] / disease_counts.iloc[-1]:.1f}x')

### Observations — class distribution

The dataset has **moderate class imbalance**: the most common disease appears significantly more often than the rarest. This matters for:
- **Training**: without correction, the model will over-predict common diseases. We apply **inverse-frequency sample weighting** so that rare diseases contribute proportionally to the loss.
- **Evaluation**: we use **Macro F1** (treats all 49 classes equally) rather than accuracy (which can be misleadingly high by always guessing the majority class).
- **Clinical relevance**: rare diseases may be exactly the ones where an automated pre-ranking tool is most helpful, since they are easiest to overlook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

train_df['AGE'].hist(bins=30, ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Patient Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

train_df['SEX'].value_counts().plot(kind='bar', ax=axes[1], color=['#4C72B0', '#DD8452'])
axes[1].set_title('Sex Distribution')
axes[1].set_xlabel('Sex')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(train_df['AGE'].describe().round(1))

In [ ]:
all_ev_flat = []
for ev_str in train_df['EVIDENCES']:
    codes = ast.literal_eval(ev_str)
    # Strip value suffixes so 'E_67_@_V_0' counts as 'E_67'
    all_ev_flat.extend(c.split('_@_')[0] for c in codes)

evidence_counts = Counter(all_ev_flat)
top_30 = pd.DataFrame(
    evidence_counts.most_common(30), columns=['code', 'count']
)
top_30['name'] = top_30['code'].map(
    lambda c: evidences.get(c, {}).get('question_en', c)[:55]
)

plt.figure(figsize=(13, 7))
sns.barplot(data=top_30, x='count', y='name', palette='viridis_r')
plt.title('Top 30 Most Frequent Symptoms in Training Set')
plt.xlabel('Occurrence Count')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
symptom_lengths = train_df['EVIDENCES'].apply(
    lambda x: len(ast.literal_eval(x))
)

plt.figure(figsize=(8, 4))
symptom_lengths.hist(bins=40, color='mediumpurple', edgecolor='white')
plt.title('Number of Symptoms Reported per Patient')
plt.xlabel('Symptom count')
plt.ylabel('Patients')
plt.tight_layout()
plt.show()

print(symptom_lengths.describe().round(1))

## 4. Feature Engineering

### Why multi-hot encoding?
Each patient's symptoms are represented as a **multi-hot (binary) vector** of length equal to the evidence vocabulary size (~223). A `1` in position `i` means evidence `i` was reported; `0` means absent. This representation:

- Is **lossless** for XGBoost: tree splits on individual binary features are equivalent to asking "is this symptom present?"
- Is **fixed-length**: every patient, regardless of how many symptoms they have, maps to the same 229-dimensional vector
- Is **interpretable**: SHAP values directly answer "how much did symptom X push the prediction toward disease Y?"

Evidence codes with value suffixes (e.g. `E_67_@_V_0` = severity level 0 of symptom 67) are collapsed to their base code (`E_67`) for simplicity. A more sophisticated encoding would preserve the ordinal value — this is left as a possible extension.

### Feature vector layout
| Slice | Size | Content |
|-------|------|----------|
| `[0 : n_codes]` | ~223 | Multi-hot symptom presence |
| `[n_codes : n_codes+5]` | 5 | One-hot age bins: 0–17, 18–35, 36–55, 56–75, 75+ |
| `[n_codes+5]` | 1 | Sex: M=1, F=0 |
| **Total** | **~229** | |

The **ordering** of features is saved to `feature_columns.pkl` — inference must use the **exact same order** or predictions will be silently wrong.

In [ ]:
all_codes = sorted(evidences.keys())
code_to_idx = {code: i for i, code in enumerate(all_codes)}
print(f'Evidence vocabulary size: {len(all_codes)} symptom codes')

In [ ]:
AGE_BINS = [0, 18, 36, 56, 76, 120]

def encode_patient(row, code_to_idx, age_bins=AGE_BINS):
    """Encode one patient row into a fixed-length float32 feature vector."""
    # Multi-hot symptom vector
    vec = np.zeros(len(code_to_idx), dtype=np.float32)
    for code in ast.literal_eval(row['EVIDENCES']):
        base_code = code.split('_@_')[0]
        if base_code in code_to_idx:
            vec[code_to_idx[base_code]] = 1.0

    # Age binning (one-hot, 5 bins)
    age = int(row['AGE'])
    age_bin = np.zeros(5, dtype=np.float32)
    for i in range(len(age_bins) - 1):
        if age_bins[i] <= age < age_bins[i + 1]:
            age_bin[i] = 1.0
            break

    # Sex (binary)
    sex = np.array([1.0 if row['SEX'] == 'M' else 0.0], dtype=np.float32)

    return np.concatenate([vec, age_bin, sex])

# Sanity-check on one row
sample_vec = encode_patient(train_df.iloc[0], code_to_idx)
print(f'Feature vector length: {len(sample_vec)}')
print(f'Non-zero features    : {int(sample_vec.sum())} (symptoms + age + sex bits)')

In [ ]:
print('Encoding training set (this takes a few minutes on 1M rows) ...')
X_train = np.vstack([encode_patient(r, code_to_idx) for _, r in train_df.iterrows()])

print('Encoding validation set ...')
X_val = np.vstack([encode_patient(r, code_to_idx) for _, r in val_df.iterrows()])

print('Encoding test set ...')
X_test = np.vstack([encode_patient(r, code_to_idx) for _, r in test_df.iterrows()])

le = LabelEncoder()
y_train = le.fit_transform(train_df['PATHOLOGY'])
y_val   = le.transform(val_df['PATHOLOGY'])
y_test  = le.transform(test_df['PATHOLOGY'])

print(f'\nFeature matrix shape : {X_train.shape}')
print(f'Number of classes    : {len(le.classes_)}')
print(f'Classes: {list(le.classes_[:5])} ...')

## 5. Model Training

### Why XGBoost?
- **Tabular data, binary features**: gradient-boosted trees are the state-of-the-art for structured/tabular data and handle binary multi-hot vectors natively.
- **Interpretability**: SHAP values for XGBoost are computed exactly (not approximated) using the TreeExplainer, giving reliable per-feature attributions.
- **Speed**: XGBoost with `n_jobs=-1` uses all CPU cores; GPU training is also supported.

### Objective: `multi:softprob`
This objective returns a **probability distribution over all 49 classes** for each patient — exactly what we need for a ranked differential diagnosis. Contrast with `multi:softmax` which returns only the argmax.

### Early stopping
We monitor validation log-loss (`mlogloss`) and stop when it hasn't improved for 20 rounds. This prevents overfitting and saves training time.

### Class weighting
Inverse-frequency sample weights upweight rare diseases so they contribute proportionally to the gradient signal, counteracting the class imbalance seen in Section 3.

In [ ]:
class_counts = Counter(y_train.tolist())
total = len(y_train)
n_classes = len(class_counts)
class_weights = {
    cls: total / (n_classes * cnt)
    for cls, cnt in class_counts.items()
}
sample_weights = np.array([class_weights[y] for y in y_train], dtype=np.float32)

print(f'Min sample weight: {sample_weights.min():.4f}')
print(f'Max sample weight: {sample_weights.max():.4f}')
print(f'Ratio            : {sample_weights.max() / sample_weights.min():.1f}x')

In [ ]:
model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    early_stopping_rounds=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f'\nBest iteration: {model.best_iteration}')
print(f'Best val log-loss: {model.best_score:.4f}')

In [ ]:
results = model.evals_result()

plt.figure(figsize=(10, 4))
plt.plot(results['validation_0']['mlogloss'], label='val log-loss', color='steelblue')
plt.axvline(
    model.best_iteration,
    color='crimson', linestyle='--',
    label=f'best iteration ({model.best_iteration})'
)
plt.title('Training Curve — Validation Log Loss')
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Evaluation

We evaluate on the **held-out test split** — data the model has never seen.

### Metrics and why each matters in a clinical context

| Metric | Why it matters |
|--------|----------------|
| **Top-1 accuracy** | Exact-match: is the highest-ranked prediction the true diagnosis? |
| **Top-3 accuracy** | Most clinically relevant: a clinician reviewing a short differential of 3 would likely catch the correct diagnosis if it appears anywhere in that list. |
| **Top-5 accuracy** | A wider safety net. |
| **Macro F1** | Averages F1 equally across all 49 classes, regardless of how common each is. Penalises the model for ignoring rare diseases. |
| **Brier score** | Measures **probability calibration**: a model with a Brier score of 0 is perfectly calibrated. A high-confidence wrong prediction is penalised more heavily than a low-confidence one. Crucial for any system where the probability itself (not just the rank) is shown to a user. |

In [ ]:
y_pred_proba = model.predict_proba(X_test)
y_pred       = np.argmax(y_pred_proba, axis=1)

top1 = top_k_accuracy_score(y_test, y_pred_proba, k=1)
top3 = top_k_accuracy_score(y_test, y_pred_proba, k=3)
top5 = top_k_accuracy_score(y_test, y_pred_proba, k=5)

print(f'Top-1 Accuracy : {top1:.4f}')
print(f'Top-3 Accuracy : {top3:.4f}  <- primary clinical metric')
print(f'Top-5 Accuracy : {top5:.4f}')

# Macro F1
report = classification_report(
    y_test, y_pred, target_names=le.classes_, output_dict=True
)
print(f"\nMacro F1       : {report['macro avg']['f1-score']:.4f}")

# Brier score (average across all classes)
n_cls = len(le.classes_)
y_test_bin = label_binarize(y_test, classes=range(n_cls))
brier = np.mean(
    [brier_score_loss(y_test_bin[:, i], y_pred_proba[:, i]) for i in range(n_cls)]
)
print(f'Mean Brier Score: {brier:.4f}  (lower = better calibrated, 0 = perfect)')

In [ ]:
report_df = pd.DataFrame(report).T
top10_diseases = train_df['PATHOLOGY'].value_counts().head(10).index.tolist()

print('Per-class metrics — top 10 most common diseases:')
display(report_df.loc[top10_diseases, ['precision', 'recall', 'f1-score']].round(4))

In [ ]:
top15 = train_df['PATHOLOGY'].value_counts().head(15).index.tolist()
top15_idx = [list(le.classes_).index(d) for d in top15]
mask = np.isin(y_test, top15_idx)

cm = confusion_matrix(y_test[mask], y_pred[mask], labels=top15_idx)

plt.figure(figsize=(15, 12))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=top15, yticklabels=top15,
    cmap='Blues'
)
plt.title('Confusion Matrix — Top 15 Diseases (Test Set)', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

### Confusion matrix analysis

The off-diagonal cells in the confusion matrix reveal which diseases the model confuses with each other. The most informative patterns to look for:

- **Respiratory cluster**: conditions like influenza, bronchitis, and viral pharyngitis share fever + cough + fatigue. The model will sometimes swap these.
- **Gastrointestinal cluster**: GERD, gastritis, and peptic ulcer disease overlap heavily on abdominal pain + nausea.
- **Musculoskeletal**: arthritis variants may share joint pain features.

These confusions are clinically expected — even experienced clinicians struggle to distinguish these conditions from symptoms alone without lab tests or imaging. The model's role here is to surface the right differential, not to make a definitive diagnosis.

**Portfolio note:** If you see a disease that is consistently misclassified as one specific other disease, it's worth checking whether those two diseases share the same top SHAP features — that would confirm the model is exploiting real clinical signal rather than spurious correlations.

## 7. Explainability with SHAP

### What is SHAP?
SHAP (SHapley Additive exPlanations) is a game-theory-based method for explaining model predictions. For each prediction, SHAP assigns a value to every feature representing **how much that feature pushed the prediction up or down** relative to the average prediction.

In plain language: if the SHAP value for "fever" in a prediction of "Influenza" is +0.15, it means that the presence of fever increased the model's log-odds for Influenza by 0.15 units. A SHAP value of 0 means the feature had no effect on this particular prediction.

For XGBoost, SHAP uses the **TreeExplainer**, which computes exact Shapley values by traversing the decision trees — no approximation needed.

### Why this matters in a medical context
A model that says "this looks like influenza with 0.72 confidence" without explanation is hard to trust. SHAP makes the model answer the follow-up question: **which specific symptoms drove that prediction?** This is the minimum viable explainability for any medical AI artifact.

> **Note:** We sample 500 test patients for SHAP computation because computing exact SHAP values for 100K+ patients and 49 classes is slow on CPU. The sample is large enough to give representative global importances.

In [ ]:
feature_names = (
    [evidences.get(c, {}).get('question_en', c)[:60] for c in all_codes]
    + [f'age_bin_{i}' for i in range(5)]
    + ['sex_male']
)
print(f'Feature names: {len(feature_names)} total')
print('Sample:', feature_names[:3], '...')

In [ ]:
sample_idx = np.random.choice(len(X_test), size=500, replace=False)
X_sample = X_test[sample_idx]

print('Computing SHAP values for 500 samples ...')
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)  # shape: (500, n_features, n_classes)

# Global mean absolute SHAP across all classes
mean_shap = np.mean(np.abs(shap_values), axis=(0, 2))
top20_idx = np.argsort(mean_shap)[-20:][::-1]

plt.figure(figsize=(11, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, 20))
plt.barh(
    [feature_names[i] for i in top20_idx],
    mean_shap[top20_idx],
    color=colors
)
plt.title('Top 20 Most Influential Symptoms (Global SHAP)', fontsize=13)
plt.xlabel('Mean |SHAP value| across all diseases')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Pick a disease and show which symptoms push predictions toward it.
# Replace 'Influenza' with any disease name from le.classes_ to explore others.
target_disease = le.classes_[0]  # use first class if 'Influenza' not present
if 'Influenza' in le.classes_:
    target_disease = 'Influenza'

target_idx = list(le.classes_).index(target_disease)
print(f'Showing SHAP summary for: {target_disease} (class index {target_idx})')

shap.summary_plot(
    shap_values[:, :, target_idx],
    X_sample,
    feature_names=feature_names,
    max_display=15,
    show=True,
    plot_type='dot',
)

### Interpreting the SHAP summary plot

In the dot plot above:
- Each dot is one of the 500 sampled patients.
- **X-axis** = SHAP value for that disease: positive = pushes prediction toward this disease, negative = pushes away.
- **Colour** = feature value: red = symptom present (1), blue = symptom absent (0).

**What to look for:**
- Features where red dots cluster on the right: these symptoms are strong positive predictors for this disease.
- Features where blue dots cluster on the right: the *absence* of this symptom paradoxically predicts this disease (unusual; worth investigating if you see it).
- Wide spread across zero: this symptom has highly variable importance — it matters a lot for some patients and not at all for others.

**Clinical sanity-check:** for Influenza, you'd expect to see fever, cough, and fatigue as the top positive predictors. If demographic features (age bins) appear near the top, that's a signal the model is leaning on population-level risk stratification rather than pure symptom content.

## 8. Saving Model Artifacts

Three files are saved to `../ml_model/artifacts/`. They are loaded by `ml_model/predict.py` at inference time:

| File | Purpose |
|------|---------|
| `xgb_model.json` | XGBoost model weights in native JSON format (portable, version-stable) |
| `label_encoder.pkl` | Maps integer class indices <-> disease name strings |
| `feature_columns.pkl` | **Ordered** list of evidence codes that defines the feature vector layout |

> **Critical:** `feature_columns.pkl` must match exactly between training and inference. If you retrain with a different vocabulary (e.g. after adding a new evidence code), you must retrain the model too — otherwise feature positions shift and predictions are silently incorrect.

In [ ]:
ARTIFACTS_DIR = Path('../ml_model/artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

model.save_model(str(ARTIFACTS_DIR / 'xgb_model.json'))
joblib.dump(le, ARTIFACTS_DIR / 'label_encoder.pkl')
joblib.dump(all_codes, ARTIFACTS_DIR / 'feature_columns.pkl')

print('Artifacts saved:')
for f in sorted(ARTIFACTS_DIR.iterdir()):
    if f.suffix in ('.json', '.pkl'):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<30} {size_kb:>8.1f} KB')

## 9. Inference Demo (End-to-End in Notebook)

This section simulates what happens at runtime when a user submits a symptom description to the main assistant:

1. `symptom_parser.py` converts free text -> structured DDXPlus feature vector
2. `ml_model/predict.py` runs the XGBoost model -> ranked disease probabilities
3. The ranked list is injected into the LLM prompt as a starting signal

Here we demonstrate step 2 directly (bypassing the NER step) using a hand-crafted feature vector.

In [ ]:
def predict_top_k(feature_vector, model, le, k=5):
    """Return the top-k ranked diseases with predicted probabilities."""
    proba = model.predict_proba(feature_vector.reshape(1, -1))[0]
    top_k_idx = np.argsort(proba)[-k:][::-1]
    return [
        {
            'disease': le.inverse_transform([i])[0],
            'probability': round(float(proba[i]), 4)
        }
        for i in top_k_idx
    ]

In [ ]:
# Build an example feature vector for a patient with a few known symptoms.
# In production, symptom_parser.py maps free text to these codes via NER + fuzzy matching.
example_features = np.zeros(len(feature_names), dtype=np.float32)

# Mark a few symptoms as present (replace with real DDXPlus codes from evidences.json)
symptom_codes_to_set = all_codes[:3]  # first 3 codes as a demo placeholder
for code in symptom_codes_to_set:
    idx = code_to_idx.get(code)
    if idx is not None:
        example_features[idx] = 1.0

# Age 28, Male -> age bin 1 (18-35) = position len(all_codes)+1; sex = last feature
example_features[len(all_codes) + 1] = 1.0  # age bin: 18-35
example_features[-1] = 1.0                   # sex: male

results = predict_top_k(example_features, model, le, k=5)
print('Top-5 predicted conditions for example patient:')
print(f'{"Disease":<40} {"Confidence"}')
print('-' * 52)
for r in results:
    print(f"{r['disease']:<40} {r['probability']:.4f}")

In [ ]:
# Visualise the ranked differential
fig, ax = plt.subplots(figsize=(9, 4))
diseases = [r['disease'] for r in results]
probs    = [r['probability'] for r in results]

colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(results))]
bars = ax.barh(diseases[::-1], probs[::-1], color=colors[::-1])
ax.set_xlabel('Predicted Probability')
ax.set_title('Ranked Differential Diagnosis (ML Classifier)')

for bar, prob in zip(bars, probs[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{prob:.3f}', va='center', fontsize=9)

ax.set_xlim(0, max(probs) * 1.2)
plt.tight_layout()
plt.show()

## 10. Summary and Limitations

### What this model does
- Predicts a **ranked differential diagnosis** (top-k disease list with probabilities) from structured symptom features extracted from free-text descriptions.
- Outputs **calibrated probabilities** (Brier score reported in Section 6) — the probability column is a real confidence estimate, not an arbitrary score.
- **Top-3 accuracy** is the primary clinical metric: in a real differential review, a short list of 3 candidates is practical and usually sufficient to prompt the right clinical path.
- Feeds its ranked predictions into the **LLM+RAG** layer as a starting signal, not a final answer — the language model still reasons from retrieved clinical passages and can override or qualify the ML output.

### Known limitations (be honest — this is especially important for a medical project)

| Limitation | Details |
|------------|----------|
| **Synthetic data** | DDXPlus was generated from a knowledge base, not real patient records. Performance on real clinical data would likely be lower due to the gap between structured, noise-free symptom codes and the messy, ambiguous language of real patient histories. |
| **Symptom parser accuracy** | The NER + fuzzy-matching layer in `symptom_parser.py` maps free text to DDXPlus codes imperfectly. Low-confidence mappings fall back to pure LLM+RAG. Any systematic mis-mapping (e.g. confusing "chest tightness" with "chest pain") introduces bias before the model even runs. |
| **49-disease scope** | DDXPlus covers 49 conditions, primarily common presentations. Rare diseases, multi-morbidity, and atypical presentations are not represented. |
| **Age/sex oversimplification** | Demographic features are coarse: age is binned into 5 groups, sex is binary. Intersectional effects (e.g. disease risk by age × sex × comorbidity) are not modelled. |
| **No temporal data** | The model treats each patient as a single cross-sectional snapshot. Duration of symptoms, progression, and prior episodes — information that significantly shifts differential probability — are not captured. |
| **No clinical validation** | This is a portfolio/educational project. It has not been prospectively validated against clinical outcomes, reviewed by clinicians, or submitted for regulatory clearance. It must not be used for clinical decision support. |

### How this notebook connects to the production system

```
This notebook (training + evaluation)
         │
         ▼ saves artifacts
ml_model/artifacts/
  xgb_model.json
  label_encoder.pkl
  feature_columns.pkl
         │
         ▼ loaded by
ml_model/predict.py  <-  symptom_parser.py  <-  assistant.py (prepare())
         │
         ▼ predictions injected into
LLM prompt  ->  Ollama  ->  FastAPI / Streamlit
```

The ML layer is **additive**: if artifacts are absent (model not yet trained), or if fewer than 3 symptoms were matched from the user's text, the system falls back gracefully to pure LLM+RAG without any error or degradation in the base system's behaviour.

---

> *This notebook is part of the Hybrid LLM+RAG+ML Medical Assistant project.*  
> *Dataset: DDXPlus (NeurIPS 2022) — research use only.*  
> *This is general information, not a medical diagnosis. Consult a qualified healthcare professional for any health concern.*